# 08b — Match Fink diaObjects with Gaia DR3 via target name (exact ID join)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Created:** 2026-04-11  
**Inspired by:** `08_matchFinkAlertswithGaiaDR3.ipynb` (cone-search version)

## Objectives

1. Load the Fink diaObject catalogue produced by `01_fink_block_flatlightcurves.ipynb`
   (`data_FINK_BLOCK_LC_01/diaobj_catalogue_gaia_stable.csv`).
   This file already carries the Fink cross-match result as a `target_name` string
   of the form `"Gaia DR3 <source_id>"`.
2. Extract the numeric Gaia DR3 `source_id` from `target_name`.
3. Load the Gaia DR3 reference catalogue
   (`../06_gaia/data_GAIA_STARCAT_DR3_03/allgaia_sources_allddf.csv`).
4. **Join by Gaia DR3 source_id** (exact integer key — no cone-search required).
5. Produce the same statistical summary and figures as in 08, adapted for the
   exact-ID join context (match completeness, Gaia stability analysis, sky maps, …).

## Data paths

| Dataset | Path |
|---------|------|
| Fink diaObject catalogue (stable groups) | `data_FINK_BLOCK_LC_01/diaobj_catalogue_gaia_stable.csv` |
| Gaia DR3 all-DDF catalogue               | `../06_gaia/data_GAIA_STARCAT_DR3_03/allgaia_sources_allddf.csv` |

## Key difference from 08

Notebook 08 uses `astropy.coordinates.match_to_catalog_sky` (cone-search, radius threshold).
**This notebook (08b) uses a pure pandas merge on `source_id`** — the association is exact
because the Fink broker already resolved the DR3 identifier when assigning the target name.
There is no astrometric tolerance to tune and no spurious matches due to nearby stars.


## 0. Imports and configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ #
# Paths
# ------------------------------------------------------------------ #
DATA_FINK = "data_FINK_BLOCK_LC_01"
DATA_GAIA = "../06_gaia/data_GAIA_STARCAT_DR3_03"
OUT_DIR = "data_FINK_BLOCK_LC_08b"
FIGS_DIR = "figs_FINK_BLOCK_LC_08b"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

# Input files
FINK_CAT_CSV = f"{DATA_FINK}/diaobj_catalogue_gaia_stable.csv"
GAIA_CAT_CSV = f"{DATA_GAIA}/allgaia_sources_allddf.csv"

print(f"Fink catalogue : {FINK_CAT_CSV}")
print(f"Gaia catalogue : {GAIA_CAT_CSV}")
print(f"Output dir     : {OUT_DIR}")
print(f"Figures dir    : {FIGS_DIR}")

## 1. Load Fink diaObject catalogue

The file `diaobj_catalogue_gaia_stable.csv` is produced by
`01_fink_block_flatlightcurves.ipynb`.  It contains one row per diaObject
with columns including `diaObjectId`, `group`, `target_name`, `ra`, `dec`,
`field`, `nDiaSources`, and several cross-match columns (`xm_gaiadr3_*`, …).

The `target_name` column stores the Fink-resolved Gaia DR3 name as a string
of the form `"Gaia DR3 <source_id>"`, e.g.
`"Gaia DR3 3836271179698567936"`.  We extract the trailing integer.

In [ ]:
df_fink = pd.read_csv(FINK_CAT_CSV)

print("Fink catalogue shape:", df_fink.shape)
print("Columns:", df_fink.columns.tolist())
print(df_fink.head(3).to_string())

In [ ]:
# ------------------------------------------------------------------ #
# Extract numeric source_id from target_name
#
# target_name format: "Gaia DR3 <integer_id>"
# We split on whitespace and take the last token, then cast to int64.
# Rows where target_name is NaN or does not contain a DR3 string
# will produce NaN after extraction (they are kept but will not match).
# ------------------------------------------------------------------ #


def extract_dr3_id(name):
    """Extract the numeric Gaia DR3 source_id from a target name string.

    Parameters
    ----------
    name : str or float (NaN)
        Fink target name, e.g. 'Gaia DR3 3836271179698567936'.

    Returns
    -------
    int or pd.NA
    """
    if pd.isna(name):
        return pd.NA
    tokens = str(name).split()
    if len(tokens) < 3:
        return pd.NA
    try:
        return int(tokens[-1])
    except ValueError:
        return pd.NA


df_fink["dr3_source_id"] = df_fink["target_name"].apply(extract_dr3_id)

n_with_id = df_fink["dr3_source_id"].notna().sum()
n_without = df_fink["dr3_source_id"].isna().sum()

print(f"Total diaObjects           : {len(df_fink)}")
print(f"  with a DR3 source_id     : {n_with_id}  ({100 * n_with_id / len(df_fink):.1f}%)")
print(f"  without (NaN target_name): {n_without}")
print()
print("Sample extracted IDs:")
print(df_fink[["diaObjectId", "target_name", "dr3_source_id", "group"]].head(5).to_string())

In [ ]:
# Distribution per Fink group
print("diaObjects per Fink group:")
print(df_fink["group"].value_counts())

## 2. Load Gaia DR3 catalogue

In [ ]:
df_gaia = pd.read_csv(GAIA_CAT_CSV)

print("Gaia DR3 catalogue shape:", df_gaia.shape)
print("Columns:", df_gaia.columns.tolist())
print(df_gaia.head(3).to_string())

In [ ]:
# Ensure source_id is integer (int64) for the join
df_gaia["source_id"] = df_gaia["source_id"].astype("int64")

# Gaia variability flag columns
VARI_COLS = [
    "in_vari_long_period_variable",
    "in_vari_eclipsing_binary",
    "in_vari_classification_result",
    "in_vari_rrlyrae",
    "in_vari_cepheid",
    "in_vari_planetary_transit",
    "in_vari_short_timescale",
    "in_vari_long_period_variable2",
    "in_vari_eclipsing_binary2",
    "in_vari_rotation_modulation",
    "in_vari_ms_oscillator",
    "in_vari_agn",
    "in_vari_microlensing",
    "in_vari_compact_companion",
]

# Keep only the columns that actually exist in the catalogue
VARI_COLS = [c for c in VARI_COLS if c in df_gaia.columns]

# Composite variability flag
df_gaia["is_gaia_variable"] = (
    df_gaia[VARI_COLS].fillna(False).astype(bool).any(axis=1) if VARI_COLS else False
)

print(
    "Gaia sources flagged as variable:",
    df_gaia["is_gaia_variable"].sum(),
    f"({100 * df_gaia['is_gaia_variable'].mean():.1f}%)",
)
if "STABLE" in df_gaia.columns:
    print("Gaia STABLE column distribution:")
    print(df_gaia["STABLE"].value_counts(dropna=False))
else:
    print("Note: 'STABLE' column not found in Gaia catalogue — skipping STABLE-based analysis.")

## 3. Join on Gaia DR3 source_id (exact match)

We perform a **left join**: every Fink diaObject is kept; Gaia columns are
populated only when the extracted `dr3_source_id` is found in the Gaia
catalogue.  A right-only Fink object (no DR3 name) will have all Gaia
columns set to NaN.

This replaces the cone-search step of notebook 08 entirely.  There is no
angular tolerance parameter and no risk of cross-matching to the wrong star.

In [ ]:
# Select Gaia columns to attach to the Fink table
GAIA_COLS_TO_PULL = [
    "source_id",
    "ra",
    "dec",
    "phot_g_mean_mag",
    "astrometric_gof_al",
    "phot_bp_rp_excess_factor",
    "std_dev_mag_g_fov",
    "DDF",
    "STABLE",
    "is_gaia_variable",
] + VARI_COLS

# Keep only columns that exist
GAIA_COLS_TO_PULL = [c for c in GAIA_COLS_TO_PULL if c in df_gaia.columns]

df_gaia_sub = df_gaia[GAIA_COLS_TO_PULL].copy()
df_gaia_sub.columns = ["gaia_" + c for c in df_gaia_sub.columns]
df_gaia_sub = df_gaia_sub.rename(columns={"gaia_source_id": "dr3_source_id"})

# Left join
df_fink["dr3_source_id"] = df_fink["dr3_source_id"].astype("Int64")  # nullable int
df_gaia_sub["dr3_source_id"] = df_gaia_sub["dr3_source_id"].astype("Int64")

df_full = df_fink.merge(df_gaia_sub, on="dr3_source_id", how="left")

# Match flag: Gaia row was found
df_full["has_gaia_match"] = (
    df_full["gaia_ra"].notna() if "gaia_ra" in df_full.columns else df_full["gaia_phot_g_mean_mag"].notna()
)

print("Merged table shape:", df_full.shape)
print(
    f"Fink objects with Gaia DR3 entry  : {df_full['has_gaia_match'].sum()} / {len(df_full)} ({100 * df_full['has_gaia_match'].mean():.1f}%)"
)
print(f"Fink objects WITHOUT Gaia DR3 entry: {(~df_full['has_gaia_match']).sum()}")

In [ ]:
# Restrict to matched subset
df_match = df_full[df_full["has_gaia_match"]].copy()
df_nomatch = df_full[~df_full["has_gaia_match"]].copy()

print("Matched diaObjects  :", len(df_match))
print("Unmatched diaObjects:", len(df_nomatch))
print()
print("Matched sample:")
print(
    df_match[["diaObjectId", "group", "target_name", "dr3_source_id", "gaia_phot_g_mean_mag", "ra", "dec"]]
    .head(3)
    .to_string()
)

## 4. Statistical summary

In [ ]:
# --- 4.1 Global match statistics ---
n_fink = len(df_full)
n_match = len(df_match)
n_no_match = len(df_nomatch)
n_no_dr3id = df_full["dr3_source_id"].isna().sum()

print("=" * 60)
print("GLOBAL MATCH STATISTICS (exact DR3 source_id join)")
print("=" * 60)
print(f"  Total Fink diaObjects              : {n_fink}")
print(f"  DR3 source_id extracted            : {n_fink - n_no_dr3id}")
print(f"  Joined to Gaia catalogue           : {n_match}  ({100 * n_match / n_fink:.1f}%)")
print(f"  Not in Gaia catalogue (ID missing) : {n_no_match}  ({100 * n_no_match / n_fink:.1f}%)")
print()

In [ ]:
# --- 4.2 Match rate per Fink group ---
grp_stats = (
    df_full.groupby("group")["has_gaia_match"]
    .agg(n_total="count", n_matched="sum")
    .assign(match_rate=lambda x: 100 * x["n_matched"] / x["n_total"])
)
print("--- Match rate per Fink group ---")
print(grp_stats.to_string())
print()

In [ ]:
# --- 4.3 Among matched objects: Fink group vs Gaia STABLE / variable flags ---
if "gaia_STABLE" in df_match.columns:
    ct_stable = pd.crosstab(df_match["group"], df_match["gaia_STABLE"], margins=True, margins_name="Total")
    print("--- Cross-tab: Fink group vs Gaia STABLE flag ---")
    print(ct_stable.to_string())
    print()

if "gaia_is_gaia_variable" in df_match.columns:
    ct_vari = pd.crosstab(
        df_match["group"], df_match["gaia_is_gaia_variable"], margins=True, margins_name="Total"
    )
    print("--- Cross-tab: Fink group vs Gaia is_variable (any flag) ---")
    print(ct_vari.to_string())

In [ ]:
# --- 4.4 Per-DDF match statistics ---
agg_dict = {"diaObjectId": "count"}
if "gaia_STABLE" in df_match.columns:
    agg_dict["gaia_STABLE"] = lambda x: x.sum()
if "gaia_is_gaia_variable" in df_match.columns:
    agg_dict["gaia_is_gaia_variable"] = lambda x: x.sum()
if "gaia_phot_g_mean_mag" in df_match.columns:
    agg_dict["gaia_phot_g_mean_mag"] = "mean"

ddf_stats = df_match.groupby("field").agg(**{k: (k, v) for k, v in agg_dict.items()})
ddf_stats.columns = ["n_objects"] + list(agg_dict.keys())[1:]
print("--- Per-DDF match statistics (matched objects only) ---")
print(ddf_stats.to_string())

In [ ]:
# --- 4.5 Individual variability flags for matched Fink objects ---
vari_flag_cols_gaia = ["gaia_" + c for c in VARI_COLS if "gaia_" + c in df_match.columns]

if vari_flag_cols_gaia:
    flag_counts = (
        df_match[vari_flag_cols_gaia]
        .fillna(False)
        .astype(bool)
        .sum()
        .rename(lambda c: c.replace("gaia_in_vari_", ""))
    )
    print("--- Gaia variability flags triggered among matched Fink objects ---")
    print(flag_counts.to_string())

if "gaia_is_gaia_variable" in df_match.columns:
    n_var = df_match["gaia_is_gaia_variable"].sum()
    print(
        f"\nObjects with >=1 Gaia vari flag: {n_var} / {len(df_match)} ({100 * n_var / len(df_match):.1f}%)"
    )

## 5. Figures

In [ ]:
# ================================================================== #
# Fig 1 — Match completeness per Fink group (bar chart)
#
# Unlike notebook 08 (which had a separation histogram here), the
# exact-ID join has no continuous separation metric.  We instead show
# the fraction of diaObjects that were successfully joined.
# ================================================================== #
grp_stats_plot = grp_stats.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
grp_stats_plot[["n_matched", "n_total"]].plot(
    kind="bar", ax=ax, color=["royalblue", "lightgrey"], edgecolor="k", linewidth=0.6, rot=15
)
ax.set_title("Match count per Fink group")
ax.set_xlabel("Fink group")
ax.set_ylabel("N diaObjects")
ax.legend(["Matched to Gaia DR3", "Total"], fontsize=9)

ax = axes[1]
grp_stats_plot["match_rate"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="k", linewidth=0.6, rot=15)
ax.set_title("Match rate per Fink group [%]")
ax.set_xlabel("Fink group")
ax.set_ylabel("Match rate [%]")
ax.set_ylim(0, 110)
ax.axhline(100, color="grey", lw=0.8, ls="--")

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig01_match_completeness_per_group.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig01")

In [ ]:
# ================================================================== #
# Fig 2 — Sky map: matched vs unmatched diaObjects
# ================================================================== #
fig, ax = plt.subplots(figsize=(10, 6))

mask_no = ~df_full["has_gaia_match"]
mask_yes = df_full["has_gaia_match"]

ax.scatter(
    df_full.loc[mask_no, "ra"],
    df_full.loc[mask_no, "dec"],
    s=14,
    c="grey",
    alpha=0.5,
    label="No Gaia DR3 entry",
    zorder=2,
)
ax.scatter(
    df_full.loc[mask_yes, "ra"],
    df_full.loc[mask_yes, "dec"],
    s=16,
    c="royalblue",
    alpha=0.8,
    label="Joined to Gaia DR3",
    zorder=3,
)

ax.set_xlabel("R.A. [deg]")
ax.set_ylabel("Dec. [deg]")
ax.set_title("Sky map of Fink diaObjects — Gaia DR3 join status (by target name)")
ax.legend(markerscale=2)
ax.invert_xaxis()
ax.grid()

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig02_skymap_match_status.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig02")

In [ ]:
# ================================================================== #
# Fig 3 — Sky map coloured by Gaia stability class
# ================================================================== #
def gaia_stability_label(row):
    """Classify a matched diaObject by Gaia photometric stability."""
    if row.get("gaia_is_gaia_variable", False):
        return "Gaia variable"
    stable_val = row.get("gaia_STABLE", None)
    if stable_val is True or stable_val == 1:
        return "Gaia stable"
    return "Gaia STABLE=False, no vari flag"


df_match = df_match.copy()
df_match["gaia_status"] = df_match.apply(gaia_stability_label, axis=1)

color_map = {
    "Gaia stable": "royalblue",
    "Gaia variable": "crimson",
    "Gaia STABLE=False, no vari flag": "darkorange",
}

fig, ax = plt.subplots(figsize=(11, 6))
for label, color in color_map.items():
    sub = df_match[df_match["gaia_status"] == label]
    if len(sub) == 0:
        continue
    ax.scatter(
        sub["ra"],
        sub["dec"],
        s=16,
        c=color,
        alpha=0.8,
        label=f"{label} (N={len(sub)})",
        zorder=3,
    )

ax.set_xlabel("R.A. [deg]")
ax.set_ylabel("Dec. [deg]")
ax.set_title("Matched Fink diaObjects — Gaia DR3 stability status")
ax.legend(markerscale=2, fontsize=9)
ax.invert_xaxis()
ax.grid()

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig03_skymap_gaia_stability.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig03")

In [ ]:
# ================================================================== #
# Fig 4 — Stacked bar: Gaia stability fraction per Fink group
# ================================================================== #
ct_status = pd.crosstab(df_match["group"], df_match["gaia_status"])
ct_frac = ct_status.div(ct_status.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ct_status.plot(
    kind="bar",
    ax=axes[0],
    color=[color_map.get(c, "grey") for c in ct_status.columns],
    edgecolor="k",
    linewidth=0.5,
    rot=20,
)
axes[0].set_title("Gaia stability — counts per Fink group")
axes[0].set_xlabel("Fink group")
axes[0].set_ylabel("N objects")
axes[0].legend(fontsize=8)

ct_frac.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=[color_map.get(c, "grey") for c in ct_frac.columns],
    edgecolor="k",
    linewidth=0.5,
    rot=20,
)
axes[1].set_title("Gaia stability — fraction per Fink group [%]")
axes[1].set_xlabel("Fink group")
axes[1].set_ylabel("Fraction [%]")
axes[1].set_ylim(0, 110)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig04_bar_gaia_stability_per_group.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig04")

In [ ]:
# ================================================================== #
# Fig 5 — Gaia G mag distribution: stable vs variable
# ================================================================== #
if "gaia_phot_g_mean_mag" in df_match.columns:
    fig, ax = plt.subplots(figsize=(8, 5))

    for label, color in color_map.items():
        sub = df_match[df_match["gaia_status"] == label]
        if len(sub) == 0:
            continue
        ax.hist(
            sub["gaia_phot_g_mean_mag"].dropna(),
            bins=30,
            histtype="stepfilled",
            alpha=0.55,
            color=color,
            edgecolor=color,
            label=f"{label} (N={len(sub)})",
        )

    ax.set_xlabel("Gaia G mean magnitude [mag]")
    ax.set_ylabel("N matched objects")
    ax.set_title("Gaia G magnitude distribution by stability class")
    ax.legend(fontsize=9)
    ax.xaxis.set_minor_locator(AutoMinorLocator())

    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig05_Gmag_stability.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig05")
else:
    print("gaia_phot_g_mean_mag not available — fig05 skipped.")

In [ ]:
# ================================================================== #
# Fig 6 — Gaia variability flags breakdown (bar chart)
# ================================================================== #
if vari_flag_cols_gaia:
    vari_subset = df_match[df_match.get("gaia_is_gaia_variable", pd.Series(False, index=df_match.index))]
    if len(vari_subset) > 0:
        flag_vals = (
            vari_subset[vari_flag_cols_gaia]
            .fillna(False)
            .astype(bool)
            .sum()
            .rename(lambda c: c.replace("gaia_in_vari_", ""))
            .sort_values(ascending=False)
        )

        fig, ax = plt.subplots(figsize=(10, 4))
        flag_vals[flag_vals > 0].plot(kind="bar", ax=ax, color="crimson", edgecolor="k", linewidth=0.5)
        ax.set_title("Gaia DR3 variability flags — Fink-matched objects flagged as variable")
        ax.set_xlabel("Gaia variability table")
        ax.set_ylabel("N objects")
        ax.tick_params(axis="x", rotation=40)
        plt.tight_layout()
        plt.savefig(f"{FIGS_DIR}/fig06_gaia_vari_flags.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved fig06")
    else:
        print("No Gaia-variable objects among matched Fink objects — fig06 skipped.")
else:
    print("No variability flag columns found — fig06 skipped.")

In [ ]:
# ================================================================== #
# Fig 7 — Astrometric offset Fink vs Gaia DR3 (diagnostic)
#
# Unlike the cone-search version, here the association is exact by ID,
# so the offset purely reflects the astrometric accuracy of the
# Fink diaObject positions relative to the Gaia DR3 reference frame.
# ================================================================== #
if "gaia_ra" in df_match.columns and "gaia_dec" in df_match.columns:
    df_match = df_match.copy()
    cos_dec = np.cos(np.radians(df_match["dec"].values))
    df_match["dra_arcsec"] = (df_match["ra"] - df_match["gaia_ra"]) * cos_dec * 3600.0
    df_match["ddec_arcsec"] = (df_match["dec"] - df_match["gaia_dec"]) * 3600.0
    df_match["sep_arcsec"] = np.hypot(df_match["dra_arcsec"], df_match["ddec_arcsec"])

    rms_ra = df_match["dra_arcsec"].std()
    rms_dec = df_match["ddec_arcsec"].std()
    med_sep = df_match["sep_arcsec"].median()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    sc = ax.scatter(
        df_match["dra_arcsec"],
        df_match["ddec_arcsec"],
        c=df_match.get("gaia_is_gaia_variable", 0).astype(int),
        cmap="coolwarm",
        s=10,
        alpha=0.6,
        vmin=0,
        vmax=1,
    )
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("Gaia variable (0=no, 1=yes)")
    cbar.set_ticks([0, 1])
    ax.axhline(0, color="k", lw=0.7, ls=":")
    ax.axvline(0, color="k", lw=0.7, ls=":")
    ax.set_xlabel(r"$\Delta\alpha\,\cos\delta$ [arcsec]")
    ax.set_ylabel(r"$\Delta\delta$ [arcsec]")
    ax.set_title("Fink − Gaia DR3 astrometric residuals (exact-ID join)")
    ax.set_aspect("equal")

    ax = axes[1]
    ax.hist(
        df_match["dra_arcsec"], bins=50, histtype="step", color="royalblue", label=r"$\Delta\alpha\cos\delta$"
    )
    ax.hist(df_match["ddec_arcsec"], bins=50, histtype="step", color="darkorange", label=r"$\Delta\delta$")
    ax.axvline(0, color="k", lw=0.7, ls=":")
    ax.set_xlabel("Residual [arcsec]")
    ax.set_ylabel("N objects")
    ax.set_title("Astrometric residual distributions")
    ax.legend()
    ax.text(
        0.97,
        0.95,
        f'RMS $\\Delta\\alpha\\cos\\delta$ = {rms_ra:.3f}"\n'
        f'RMS $\\Delta\\delta$ = {rms_dec:.3f}"\n'
        f'Median sep = {med_sep:.3f}"',
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8),
    )

    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig07_astrometric_residuals.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig07")
else:
    print("Gaia (ra, dec) columns not available — fig07 skipped.")

In [ ]:
# ================================================================== #
# Fig 8 — nDiaSources distribution: matched vs unmatched
#
# Checks whether the few unmatched diaObjects (DR3 ID not in catalogue)
# are sparse objects (few alerts) or well-observed ones.
# ================================================================== #
fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(0, df_full["nDiaSources"].quantile(0.99) + 1, 40)

ax.hist(
    df_full.loc[df_full["has_gaia_match"], "nDiaSources"],
    bins=bins,
    histtype="stepfilled",
    alpha=0.6,
    color="royalblue",
    edgecolor="royalblue",
    label=f"Joined to Gaia DR3 (N={df_full['has_gaia_match'].sum()})",
)
ax.hist(
    df_full.loc[~df_full["has_gaia_match"], "nDiaSources"],
    bins=bins,
    histtype="stepfilled",
    alpha=0.6,
    color="crimson",
    edgecolor="crimson",
    label=f"Not in Gaia catalogue (N={(~df_full['has_gaia_match']).sum()})",
)

ax.set_xlabel("nDiaSources (number of Fink alerts)")
ax.set_ylabel("N diaObjects")
ax.set_title("Alert count distribution: joined vs not-joined objects")
ax.legend(fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig08_nDiaSources_match_status.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig08")

## 6. Save outputs

In [ ]:
# Full table (all Fink objects + Gaia columns where joined)
df_full.to_csv(f"{OUT_DIR}/fink_diaobj_gaia_join_all.csv", index=False)
print(f"Saved: {OUT_DIR}/fink_diaobj_gaia_join_all.csv  ({len(df_full)} rows)")

# Matched-only table
df_match.to_csv(f"{OUT_DIR}/fink_diaobj_gaia_join_matched.csv", index=False)
print(f"Saved: {OUT_DIR}/fink_diaobj_gaia_join_matched.csv  ({len(df_match)} rows)")

# Unmatched objects (DR3 ID not found in catalogue)
df_nomatch.to_csv(f"{OUT_DIR}/fink_diaobj_gaia_join_unmatched.csv", index=False)
print(f"Saved: {OUT_DIR}/fink_diaobj_gaia_join_unmatched.csv  ({len(df_nomatch)} rows)")

In [ ]:
# Quick display of matched table
df_match

## 7. Summary and conclusions

This notebook performed an **exact DR3 source_id join** between the Fink
diaObject catalogue (`diaobj_catalogue_gaia_stable.csv`, produced by
`01_fink_block_flatlightcurves.ipynb`) and the Gaia DR3 reference catalogue
covering all DDFs.

### Methodology vs notebook 08

| Criterion | 08 (cone-search) | **08b (exact ID join)** |
|-----------|-----------------|------------------------|
| Matching algorithm | `astropy match_to_catalog_sky` | `pandas.merge` on `source_id` |
| Tolerance parameter | `MATCH_RADIUS_ARCSEC` | none |
| Risk of mis-match | possible for crowded fields | zero (same DR3 identifier) |
| Prerequisite | diaObject (ra, dec) | Fink has resolved the DR3 name |
| Input file | `gaia_star_stable_src.parquet` | `diaobj_catalogue_gaia_stable.csv` |

### Key results

- Match rates per Fink group are reported in Section 4.2.
- The `gaia_STABLE` flag and the composite `is_gaia_variable` flag
  (union of all Gaia variability tables) are cross-tabulated against
  the Fink group in Sections 4.3 and Figures 3/4.
- Astrometric residuals (Figure 7) characterise the positional accuracy of
  the Fink diaObject positions relative to the Gaia DR3 reference frame.
  Because the association is exact by ID, these offsets are purely
  astrometric (no cone-search contamination from neighbours).
- Figure 8 shows whether unmatched diaObjects tend to have fewer alerts,
  diagnosing potential catalogue edge effects.

### Outputs saved

| File | Content |
|------|---------|
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_all.csv`       | All Fink objects + Gaia columns (NaN where no match) |
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv`   | Matched objects only, full Gaia columns              |
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_unmatched.csv` | Objects not found in Gaia catalogue                  |
| `figs_FINK_BLOCK_LC_08b/fig01_*` … `fig08_*`                | Diagnostic figures                                    |
